# ECC3479 notebook

**Matched Tables Using House Prices + ABS Population**

Create two suburb-level tables for 2015 to 2024 using:
- `data/clean/suburbs_house_prices_2015_2024_wide_complete.csv`
- `data/raw/abs_population.xlsx` (Table 1, SA2 population)

The merge keeps all suburbs in the house prices file and brings in matching ABS SA2-name rows where available. Suburbs not included in the analysis were those that did not appear in either of the datasets.

In [48]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

house_file = project_root / "data" / "clean" / "suburbs_house_prices_2015_2024_wide_complete.csv"
abs_file = project_root / "data" / "raw" / "abs_population.xlsx"

years = [str(y) for y in range(2015, 2025)]

# ----------------------------
# 1) House prices table
# ----------------------------
house = pd.read_csv(house_file)
house = house.rename(columns={"Suburb": "suburb"})
house["suburb"] = (
    house["suburb"]
    .astype(str)
    .str.strip()
    .str.replace(r"\\s+", " ", regex=True)
    .str.upper()
    )

missing_house_years = [y for y in years if y not in house.columns]
if missing_house_years:
    raise ValueError(f"Missing house-price year columns: {missing_house_years}")

for y in years:
    house[y] = pd.to_numeric(house[y], errors="coerce")

table_house_prices = (
    house[["suburb"] + years]
    .drop_duplicates(subset=["suburb"], keep="first")
    .sort_values("suburb")
    .reset_index(drop=True)
)

# ----------------------------
# 2) Population table from ABS Table 1
# ----------------------------
# ABS Table 1 contains metadata rows; the real headers are around rows 5-6.
pop_raw = pd.read_excel(abs_file, sheet_name="Table 1", header=None)

header_top = pop_raw.iloc[4]
header_bottom = pop_raw.iloc[5]
combined_cols = []
for top, bottom in zip(header_top, header_bottom):
    if pd.notna(top) and str(top).strip().lower() != "nan":
        combined_cols.append(str(top).strip())
    else:
        combined_cols.append(str(bottom).strip())

pop = pop_raw.iloc[6:].copy()
pop.columns = combined_cols

needed_cols = ["SA2 name"] + years
missing_pop_cols = [c for c in needed_cols if c not in pop.columns]
if missing_pop_cols:
    raise ValueError(f"Missing ABS columns: {missing_pop_cols}")

pop = pop[needed_cols].rename(columns={"SA2 name": "suburb"})
pop["suburb"] = (
    pop["suburb"]
    .astype(str)
    .str.strip()
    .str.replace(r"\\s+", " ", regex=True)
    .str.upper()
    )

for y in years:
    pop[y] = pd.to_numeric(pop[y], errors="coerce")

pop = (
    pop
    .dropna(subset=["suburb"])
    .drop_duplicates(subset=["suburb"], keep="first")
    .reset_index(drop=True)
)

# Keep all house-price suburbs, bring matching ABS values where suburb names align.
table_population_density = (
    table_house_prices[["suburb"]]
    .merge(pop, on="suburb", how="left")
    .sort_values("suburb")
    .reset_index(drop=True)
)

# Remove suburbs from table_population_density that have NaN for all year columns
# Then also remove those suburbs from table_house_prices to keep both tables aligned
rows_with_all_nans = table_population_density[years].isna().all(axis=1)
suburbs_to_remove = table_population_density.loc[rows_with_all_nans, "suburb"].tolist()

table_population_density = table_population_density[~rows_with_all_nans].reset_index(drop=True)
table_house_prices = table_house_prices[~table_house_prices["suburb"].isin(suburbs_to_remove)].reset_index(drop=True)

# Keep only the chosen suburbs that exist in both tables (expected: 172 suburbs).
chosen_suburbs = sorted(set(table_house_prices["suburb"]).intersection(set(table_population_density["suburb"])))
table_house_prices = table_house_prices[table_house_prices["suburb"].isin(chosen_suburbs)].sort_values("suburb").reset_index(drop=True)
table_population_density = table_population_density[table_population_density["suburb"].isin(chosen_suburbs)].sort_values("suburb").reset_index(drop=True)

# Optional: save outputs
house_out = project_root / "data" / "clean" / "matched_houses_population_2015_2024_prices.csv"
pop_out = project_root / "data" / "clean" / "matched_houses_population_2015_2024_population.csv"
table_house_prices.to_csv(house_out, index=False)
table_population_density.to_csv(pop_out, index=False)

matched_count = len(table_population_density)
print(f"House-price suburbs total (Table 1): {len(table_house_prices)}")
print(f"Population suburbs total (Table 2): {len(table_population_density)}")
print(f"Chosen suburbs kept in both tables: {len(chosen_suburbs)}")
print(f"Removed suburbs with no ABS data: {len(suburbs_to_remove)}")
print(f"Saved: {house_out}")
print(f"Saved: {pop_out}")

print("\nTable 1: Suburbs with matching house prices (2015-2024)")
display(table_house_prices)

print("\nTable 2: Suburbs with matching ABS population values (2015-2024)")
print("(Named as requested: population density table)")
display(table_population_density)

House-price suburbs total (Table 1): 172
Population suburbs total (Table 2): 172
Chosen suburbs kept in both tables: 172
Removed suburbs with no ABS data: 314
Saved: c:\Users\samantha\Desktop\uni ૮₍•᷄ ࡇ•᷅₎ა\Y3S1\ECC3479\ECC3479-project\ecc3479-project\ECC3479-project (data + GitHub repo)\data\clean\matched_houses_population_2015_2024_prices.csv
Saved: c:\Users\samantha\Desktop\uni ૮₍•᷄ ࡇ•᷅₎ა\Y3S1\ECC3479\ECC3479-project\ecc3479-project\ECC3479-project (data + GitHub repo)\data\clean\matched_houses_population_2015_2024_population.csv

Table 1: Suburbs with matching house prices (2015-2024)


,suburb,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,ABBOTSFORD,925000.0,1187500.0,1280000.0,1192500.0,1050000.0,1200000.0,1365000.0,1346000.0,1257500.0,1232500.0
1,AIRPORT WEST,635000.0,742000.0,845000.0,845000.0,795000.0,812500.0,960000.0,911000.0,952500.0,940000.0
2,ALBERT PARK,1700000.0,1775000.0,2150000.0,2070000.0,1990000.0,1900000.0,2355000.0,2475000.0,2287500.0,2225000.0
3,ALBION,481000.0,593000.0,730000.0,755000.0,675000.0,727500.0,832500.0,830000.0,765000.0,735000.0
4,ALEXANDRA,285000.0,275000.0,291500.0,335000.0,325000.0,380000.0,440000.0,550000.0,477500.0,512500.0
5,ALFREDTON,340500.0,375000.0,412500.0,439000.0,480000.0,512000.0,605000.0,650000.0,640000.0,605000.0
6,ALTONA,718000.0,832500.0,950000.0,932500.0,890000.0,914000.0,1162500.0,1180000.0,1142500.0,1170000.0
7,ALTONA MEADOWS,440000.0,495000.0,626500.0,631500.0,615000.0,640000.0,720000.0,740000.0,690000.0,717000.0
8,ALTONA NORTH,642500.0,725000.0,825000.0,835000.0,790000.0,832500.0,900500.0,940000.0,881000.0,917000.0
9,ARARAT,210000.0,204500.0,199000.0,199000.0,215000.0,244000.0,315000.0,380000.0,375000.0,380000.0



Table 2: Suburbs with matching ABS population values (2015-2024)
(Named as requested: population density table)


,suburb,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,ABBOTSFORD,8078.0,8770.0,9291.0,9527.0,9594.0,9672.0,9258.0,9450.0,9948.0,10207.0
1,AIRPORT WEST,7694.0,7903.0,8019.0,8169.0,8390.0,8362.0,8240.0,8273.0,8437.0,8612.0
2,ALBERT PARK,16053.0,16490.0,16500.0,16728.0,17081.0,16955.0,16011.0,16126.0,16811.0,17094.0
3,ALBION,2908.0,2974.0,3328.0,3663.0,3929.0,4192.0,4371.0,4541.0,4702.0,4813.0
4,ALEXANDRA,6404.0,6474.0,6550.0,6646.0,6687.0,6690.0,6771.0,6785.0,6820.0,6836.0
5,ALFREDTON,11039.0,11852.0,12649.0,13537.0,14434.0,15507.0,16841.0,17988.0,18966.0,20082.0
6,ALTONA,12979.0,13206.0,13443.0,13657.0,13861.0,13878.0,13655.0,13711.0,13877.0,14190.0
7,ALTONA MEADOWS,19818.0,20106.0,20164.0,20101.0,20031.0,19500.0,18660.0,18334.0,18604.0,18733.0
8,ALTONA NORTH,14519.0,14787.0,14999.0,15185.0,15400.0,15396.0,15084.0,15034.0,15342.0,15851.0
9,ARARAT,8324.0,8386.0,8405.0,8428.0,8465.0,8552.0,8470.0,8375.0,8311.0,8288.0


**Classification: Status-based treated/untreated groups (from Daniel Bowen's November 2022 update)**

Suburbs are classified based on their status in the level crossing removal program:
- Treated (1.0): Status is "Completed", "Planning", or "Underway"
- Untreated (0.0): Status is blank or not listed

In [4]:
from pathlib import Path
import re
from io import StringIO
import pandas as pd
import requests

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

# Daniel Bowen source page (Nov 2022 update)
url = "https://danielbowen.com/2022/11/05/level-crossings-nov-2022-update/"

# Use a browser-like User-Agent to avoid 403 responses from default urllib headers.
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}
response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()
tables = pd.read_html(StringIO(response.text))

def normalize_col_name(col: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(col).strip().lower())

def pick_suburb_status_table(html_tables: list[pd.DataFrame]) -> tuple[pd.DataFrame, str, str]:
    candidates = []
    for t in html_tables:
        df = t.copy()

        # Flatten MultiIndex columns if present.
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [" ".join([str(c) for c in col if str(c) != "nan"]).strip() for col in df.columns]

        col_map = {c: normalize_col_name(c) for c in df.columns}
        suburb_col = None
        status_col = None

        for original, normalized in col_map.items():
            if normalized in {"suburb", "suburbs"}:
                suburb_col = original
            if normalized == "status112022":
                status_col = original

        if suburb_col is not None and status_col is not None:
            candidates.append((df, suburb_col, status_col))

    if not candidates:
        raise ValueError("Could not find table with both suburb/suburbs and status 11/2022 columns.")

    # Use the largest matching table as the main dataset.
    return max(candidates, key=lambda item: len(item[0]))

df_raw, suburb_col, status_col = pick_suburb_status_table(tables)

# Extract data ONLY from the suburb/suburbs and status columns.
df_bowen = df_raw[[suburb_col, status_col]].copy()
df_bowen.columns = ["suburb", "status"]

# Clean text and remove blank suburbs.
df_bowen["suburb"] = df_bowen["suburb"].fillna("").astype(str).str.strip().str.lower()
df_bowen["status"] = df_bowen["status"].fillna("").astype(str).str.strip().str.lower()
df_bowen = df_bowen[df_bowen["suburb"].ne("")].copy()

# Resolve duplicates by keeping the strongest status for each suburb.
status_priority = {"completed": 4, "underway": 3, "planning": 2, "": 1}
df_bowen["priority"] = df_bowen["status"].map(lambda s: status_priority.get(s, 1))

df_bowen_suburb = (
    df_bowen.sort_values("priority", ascending=False)
            .drop_duplicates(subset=["suburb"], keep="first")
            .loc[:, ["suburb", "status"]]
            .sort_values("suburb")
            .reset_index(drop=True)
)

# Treated definition from Daniel Bowen status table.
treated_statuses = {"completed", "planning", "underway"}
df_bowen_suburb["treated"] = df_bowen_suburb["status"].apply(
    lambda s: 1.0 if s in treated_statuses else 0.0
)

treated_bowen = df_bowen_suburb[df_bowen_suburb["treated"] == 1.0].reset_index(drop=True)
untreated_bowen = df_bowen_suburb[df_bowen_suburb["treated"] == 0.0].reset_index(drop=True)

clean_dir = project_root / "data" / "clean"
clean_dir.mkdir(parents=True, exist_ok=True)
treated_output = clean_dir / "daniel_bowen_treated_suburbs_2022.csv"
untreated_output = clean_dir / "daniel_bowen_untreated_suburbs_2022.csv"

treated_bowen.to_csv(treated_output, index=False)
untreated_bowen.to_csv(untreated_output, index=False)

print("Daniel Bowen — classification using Suburb/Suburbs + Status 11/2022 only")
print(f"Total unique suburbs from the page: {len(df_bowen_suburb)}")
print(f"Treated (completed/planning/underway): {len(treated_bowen)}")
print(f"Untreated (other/blank): {len(untreated_bowen)}")
print("\nTREATED suburbs (treated = 1.0):")
display(treated_bowen)
print(f"Saved treated table to: {treated_output}")
print("\nUNTREATED suburbs (treated = 0.0):")
display(untreated_bowen)
print(f"Saved untreated table to: {untreated_output}")

Daniel Bowen — classification using Suburb/Suburbs + Status 11/2022 only
Total unique suburbs from the page: 179
Treated (completed/planning/underway): 86
Untreated (other/blank): 93

TREATED suburbs (treated = 1.0):


,suburb,status,treated
0,alphington,completed,1.0
1,altona,completed,1.0
2,ardeer,underway,1.0
3,aspendale,planning,1.0
4,bayswater,completed,1.0
...,...,...,...
81,wallace,completed,1.0
82,werribee,completed,1.0
83,williamstown,completed,1.0
84,wodonga,completed,1.0


Saved treated table to: c:\Users\samantha\Desktop\uni ૮₍•᷄ ࡇ•᷅₎ა\Y3S1\ECC3479\ECC3479-project\ecc3479-project\ECC3479-project (data + GitHub repo)\data\clean\daniel_bowen_treated_suburbs_2022.csv

UNTREATED suburbs (treated = 0.0):


,suburb,status,treated
0,ararat,,0.0
1,avenel,,0.0
2,bacchus marsh,,0.0
3,ballarat,,0.0
4,baxter,,0.0
...,...,...,...
88,wendouree,,0.0
89,westmere,,0.0
90,windsor,,0.0
91,woodend,,0.0


Saved untreated table to: c:\Users\samantha\Desktop\uni ૮₍•᷄ ࡇ•᷅₎ა\Y3S1\ECC3479\ECC3479-project\ecc3479-project\ECC3479-project (data + GitHub repo)\data\clean\daniel_bowen_untreated_suburbs_2022.csv
